# 03 — Prévisions (Forecasting)
## Coût de la vie et inflation au Sénégal

On prévoit l'**IHPC national** et le **coût du panier** sur 12 mois (juin 2026 →
mai 2027) et on compare plusieurs modèles :
- **Baseline** : moyenne mobile / marche aléatoire saisonnière
- **Régression linéaire** avec tendance + saisonnalité (mois)
- **SARIMA** (statsmodels)
- **Prophet** (si la librairie est installée)

Sélection du meilleur modèle par **RMSE / MAE** sur les 12 derniers mois (test).


In [1]:

import os, warnings, pathlib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["figure.dpi"] = 110

PROJ = os.getcwd()
if not os.path.isdir(os.path.join(PROJ, "data")):
    PROJ = os.path.dirname(PROJ)
RAW = os.path.join(PROJ, "data", "raw")
PROC = os.path.join(PROJ, "data", "processed")
FIG = os.path.join(PROJ, "reports", "figures")
MODELS = os.path.join(PROJ, "models")
for d in (PROC, FIG, MODELS):
    os.makedirs(d, exist_ok=True)
print("Racine projet :", PROJ)


Racine projet : C:\projet\Bi5


In [2]:

ihpc_nat = pd.read_csv(os.path.join(PROC, "ihpc_national.csv"), parse_dates=["date"])
panier_nat = pd.read_csv(os.path.join(PROC, "panier_national.csv"), parse_dates=["date"])
serie = ihpc_nat.set_index("date")["indice_global"].asfreq("MS")
print("Série IHPC :", serie.index.min().date(), "->", serie.index.max().date(), "|", len(serie), "points")

H = 12  # horizon de test et de prévision
train, test = serie.iloc[:-H], serie.iloc[-H:]

def metrics(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    rmse = float(np.sqrt(np.mean((y-yhat)**2)))
    mae = float(np.mean(np.abs(y-yhat)))
    mape = float(np.mean(np.abs((y-yhat)/y))*100)
    return rmse, mae, mape


Série IHPC : 2018-01-01 -> 2026-05-01 | 101 points


### 1. Baseline — marche aléatoire saisonnière

In [3]:

# prévision = valeur du même mois 12 mois plus tôt
snaive = train.iloc[-12:].values[:len(test)]
res_snaive = metrics(test.values, snaive)
print("Seasonal naive  RMSE=%.2f  MAE=%.2f  MAPE=%.2f%%" % res_snaive)


Seasonal naive  RMSE=2.02  MAE=1.86  MAPE=1.79%


### 2. Régression linéaire (tendance + saisonnalité)

In [4]:

from sklearn.linear_model import LinearRegression

def make_features(idx):
    t = np.arange(len(idx))
    months = pd.get_dummies(idx.month, prefix="m", drop_first=True).reset_index(drop=True)
    X = pd.concat([pd.Series(t, name="t"), months], axis=1)
    return X

full_idx = serie.index
Xall = make_features(full_idx)
Xtr, Xte = Xall.iloc[:-H], Xall.iloc[-H:]
lr = LinearRegression().fit(Xtr, train.values)
pred_lr = lr.predict(Xte)
res_lr = metrics(test.values, pred_lr)
print("Régression lin. RMSE=%.2f  MAE=%.2f  MAPE=%.2f%%" % res_lr)


Régression lin. RMSE=3.07  MAE=2.55  MAPE=2.49%


### 3. SARIMA

In [5]:

from statsmodels.tsa.statespace.sarimax import SARIMAX
sar = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,0,12),
              enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred_sar = sar.forecast(H)
res_sar = metrics(test.values, pred_sar.values)
print("SARIMA          RMSE=%.2f  MAE=%.2f  MAPE=%.2f%%" % res_sar)


SARIMA          RMSE=0.77  MAE=0.61  MAPE=0.60%


### 4. Prophet (optionnel)

In [6]:

# Best-effort : sous Windows, le backend Stan de Prophet a besoin d'un runtime
# mingw compatible. On expose au besoin les DLL (Git for Windows + tbb cmdstan).
def _prep_prophet_runtime():
    import sys, glob
    if not sys.platform.startswith("win"):
        return
    cands = [r"C:\rtools44\x86_64-w64-mingw32.static.posix\bin",
             r"C:\rtools44\usr\bin",
             r"C:\Program Files\Git\mingw64\bin"]
    cands += glob.glob(os.path.join(os.path.dirname(os.__file__), "..", "site-packages",
                       "prophet", "stan_model", "cmdstan-*", "stan", "lib", "stan_math", "lib", "tbb"))
    for d in cands:
        if os.path.isdir(d):
            try: os.add_dll_directory(d)
            except Exception: pass
            os.environ["PATH"] = d + os.pathsep + os.environ.get("PATH", "")

res_prophet = None
pred_prophet = None
try:
    _prep_prophet_runtime()
    from prophet import Prophet
    dfp = train.reset_index(); dfp.columns = ["ds", "y"]
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m.fit(dfp)
    fut = m.make_future_dataframe(periods=H, freq="MS")
    fc = m.predict(fut)
    pred_prophet = fc.set_index("ds")["yhat"].iloc[-H:]
    res_prophet = metrics(test.values, pred_prophet.values)
    print("Prophet         RMSE=%.2f  MAE=%.2f  MAPE=%.2f%%" % res_prophet)
except Exception as e:
    print("⚠️ Prophet installé mais backend Stan non exécutable ici (%s)." % type(e).__name__)
    print("   -> Sur Windows, installer le compilateur (conda-forge `prophet`, ou")
    print("      `python -m cmdstanpy.install_cmdstan --compiler`). Le code reste")
    print("      valide et s'exécutera sur un environnement doté de la toolchain.")


Importing plotly failed. Interactive plots will not work.


22:26:05 - cmdstanpy - INFO - Chain [1] start processing


22:26:05 - cmdstanpy - INFO - Chain [1] done processing


22:26:05 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


⚠️ Prophet installé mais backend Stan non exécutable ici (RuntimeError).
   -> Sur Windows, installer le compilateur (conda-forge `prophet`, ou
      `python -m cmdstanpy.install_cmdstan --compiler`). Le code reste
      valide et s'exécutera sur un environnement doté de la toolchain.


### 5. Comparaison des modèles

In [7]:

rows = [("Seasonal naive", *res_snaive),
        ("Régression linéaire", *res_lr),
        ("SARIMA", *res_sar)]
if res_prophet:
    rows.append(("Prophet", *res_prophet))
comp = pd.DataFrame(rows, columns=["modele","RMSE","MAE","MAPE_%"]).sort_values("RMSE")
comp.to_csv(os.path.join(MODELS, "model_comparison.csv"), index=False, encoding="utf-8-sig")
best = comp.iloc[0]["modele"]
print(comp.to_string(index=False))
print("\nMeilleur modèle (RMSE) :", best)


             modele     RMSE      MAE   MAPE_%
             SARIMA 0.774818 0.609554 0.596901
     Seasonal naive 2.020139 1.856135 1.785682
Régression linéaire 3.070576 2.545875 2.486995

Meilleur modèle (RMSE) : SARIMA


### 6. Prévision 12 mois sur l'ensemble complet (meilleur modèle = SARIMA)

In [8]:

# On réentraîne SARIMA sur toute la série pour prévoir l'avenir.
final = SARIMAX(serie, order=(1,1,1), seasonal_order=(1,1,0,12),
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
fc = final.get_forecast(H)
mean = fc.predicted_mean
ci = fc.conf_int(alpha=0.20)
forecast = pd.DataFrame({"date": mean.index, "indice_prevu": mean.values,
                         "borne_basse": ci.iloc[:,0].values, "borne_haute": ci.iloc[:,1].values})
# inflation prévue (glissement annuel) en raccordant à l'historique
hist = serie.copy()
ext = pd.concat([hist, mean])
forecast["inflation_prevue_pct"] = [ (ext.loc[d]/ext.loc[d - pd.DateOffset(years=1)] - 1)*100
                                     for d in mean.index ]
forecast.to_csv(os.path.join(MODELS, "forecast_ihpc.csv"), index=False, encoding="utf-8-sig")
forecast.round(2)


,date,indice_prevu,borne_basse,borne_haute,inflation_prevue_pct
0,2026-06-01,105.35,104.75,105.96,0.81
1,2026-07-01,105.84,104.76,106.92,0.71
2,2026-08-01,105.78,104.23,107.33,0.63
3,2026-09-01,105.61,103.62,107.61,0.59
4,2026-10-01,105.36,102.94,107.77,0.53
5,2026-11-01,104.99,102.18,107.80,0.52
6,2026-12-01,102.65,99.47,105.83,0.54
7,2027-01-01,100.33,96.80,103.86,1.06
8,2027-02-01,101.41,97.55,105.27,0.93
9,2027-03-01,102.92,98.75,107.09,0.80


In [9]:

fig, ax = plt.subplots()
ax.plot(serie.index, serie.values, color="#1f4e79", lw=1.8, label="Historique")
ax.plot(forecast["date"], forecast["indice_prevu"], color="#c0392b", lw=2, label="Prévision SARIMA")
ax.fill_between(forecast["date"], forecast["borne_basse"], forecast["borne_haute"],
                color="#c0392b", alpha=.15, label="IC 80%")
ax.set_title("Prévision de l'IHPC national (12 mois)"); ax.set_ylabel("IHPC (base 100=2023)")
ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG, "12_forecast_ihpc.png"), bbox_inches="tight")
plt.close(fig); print("→ 12_forecast_ihpc.png")
print("Inflation prévue moyenne (12 prochains mois) : %.1f%%" % forecast["inflation_prevue_pct"].mean())


→ 12_forecast_ihpc.png
Inflation prévue moyenne (12 prochains mois) : 0.7%


### 7. Prévision du coût du panier de base

In [10]:

pan = panier_nat.set_index("date")["cout_panier"].asfreq("MS")
mp = SARIMAX(pan, order=(1,1,1), seasonal_order=(1,1,0,12),
             enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
fp = mp.get_forecast(H); pm = fp.predicted_mean; pci = fp.conf_int(alpha=0.20)
pan_fc = pd.DataFrame({"date": pm.index, "cout_prevu": pm.values,
                       "borne_basse": pci.iloc[:,0].values, "borne_haute": pci.iloc[:,1].values})
pan_fc.to_csv(os.path.join(MODELS, "forecast_panier.csv"), index=False, encoding="utf-8-sig")

fig, ax = plt.subplots()
ax.plot(pan.index, pan.values/1000, color="#8e44ad", lw=1.8, label="Historique")
ax.plot(pan_fc["date"], pan_fc["cout_prevu"]/1000, color="#c0392b", lw=2, label="Prévision")
ax.fill_between(pan_fc["date"], pan_fc["borne_basse"]/1000, pan_fc["borne_haute"]/1000,
                color="#c0392b", alpha=.15)
ax.set_title("Prévision du coût du panier de base (12 mois)")
ax.set_ylabel("Coût (milliers FCFA / mois)"); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG, "13_forecast_panier.png"), bbox_inches="tight")
plt.close(fig); print("→ 13_forecast_panier.png")
print("Coût panier actuel : %.0f FCFA | prévu dans 12 mois : %.0f FCFA"
      % (pan.iloc[-1], pan_fc["cout_prevu"].iloc[-1]))


→ 13_forecast_panier.png
Coût panier actuel : 139259 FCFA | prévu dans 12 mois : 140289 FCFA


### Conclusion forecasting
- Le modèle **SARIMA** capture la tendance et la saisonnalité ; il est retenu
  comme meilleur compromis (RMSE le plus faible sur le jeu de test).
- La trajectoire prévue indique une inflation **modérée et maîtrisée** à court
  terme, cohérente avec la décélération observée depuis 2023.
- ⚠️ Prévisions à interpréter avec prudence : elles reposent sur la dynamique
  passée et n'intègrent pas les chocs exogènes (prix mondiaux, mesures
  gouvernementales, climat).
